<a href="https://colab.research.google.com/github/bogdangradinaru/ai-safety/blob/main/notebooks/part-1/07-model-architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring CNN Architectures

An optional hands-on companion to [Section 3.5](../../src/part-1/03-model-architectures.md#architecture-families). This lab uses a tiny synthetic image to inspect convolution, activation, pooling, and tensor shapes in a CNN. It does not download data or use a pretrained model.

**Environment:** Python with PyTorch and Matplotlib. Google Colab includes both; for a local environment, install `torch` and `matplotlib` first.

## Learning objectives

- See how a convolution filter produces a feature map.
- Inspect how ReLU and max pooling change activations and spatial size.
- Trace tensor shapes and count the parameters in a small CNN.

Run the cells in order. The image and filter are deliberately simple so the operations are easy to inspect.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(7)
print(f"PyTorch {torch.__version__}")

## 1. Create a small input image

The 16 x 16 grayscale scene below is drawn from simple pixel values. It resembles a road and a vehicle, but it is synthetic rather than a camera image.

In [ ]:
image = torch.zeros((16, 16), dtype=torch.float32)
image[:5, :] = 0.75  # sky
image[5:, :] = 0.20  # road
image[8:12, 4:12] = 0.82  # vehicle body
image[6:8, 6:10] = 0.60  # windscreen
image[12:14, 4:6] = 0.05  # wheels
image[12:14, 10:12] = 0.05

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("Synthetic 16 x 16 scene")
plt.axis("off");

## 2. Apply a convolution filter

A CNN applies the same small kernel at each image location. This fixed edge filter is for demonstration; trained CNNs learn their filter weights from data. `conv2d` expects input shaped as batch x channels x height x width, so add batch and channel dimensions.

In [ ]:
edge_kernel = torch.tensor(
    [[[-1.0, 0.0, 1.0],
      [-2.0, 0.0, 2.0],
      [-1.0, 0.0, 1.0]]]
).unsqueeze(0)  # output channels x input channels x kernel height x kernel width
image_batch = image[None, None, :, :]

feature_map = F.conv2d(image_batch, edge_kernel, padding=1)
print("Input shape:     ", tuple(image_batch.shape))
print("Kernel shape:    ", tuple(edge_kernel.shape))
print("Feature map shape:", tuple(feature_map.shape))

plt.figure(figsize=(10, 3))
for index, (values, title) in enumerate(
    [(image, "Input"), (feature_map[0, 0], "Signed feature map")], start=1
):
    plt.subplot(1, 2, index)
    plt.imshow(values, cmap="coolwarm", vmin=-2, vmax=2)
    plt.title(title)
    plt.axis("off")
plt.tight_layout();

## 3. Add ReLU and max pooling

ReLU sets negative responses to zero. A 2 x 2 max-pooling operation keeps the largest value in each region, halving height and width in this example.

In [ ]:
activated = F.relu(feature_map)
pooled = F.max_pool2d(activated, kernel_size=2, stride=2)
print("After ReLU:     ", tuple(activated.shape))
print("After max pool: ", tuple(pooled.shape))

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, values, title in zip(
    axes,
    [feature_map[0, 0], activated[0, 0], pooled[0, 0]],
    ["Convolution", "After ReLU", "After 2 x 2 max pool"],
):
    ax.imshow(values, cmap="viridis")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout();

## 4. Trace a small CNN

This model adds a second convolution and a small output layer. The three output values are arbitrary scores: this notebook does not train the network or assign real object classes.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(4, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(8, 3),
)

x = image_batch
for layer in model:
    x = layer(x)
    print(f"{layer.__class__.__name__:>16}: {tuple(x.shape)}")

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"\nTrainable parameters: {parameter_count:,}")

## Try it yourself

1. Change the convolution's `padding` from 1 to 0. What happens to the feature-map dimensions?
2. Change max pooling to `kernel_size=2, stride=1`. How much spatial detail remains?
3. Add another convolution layer to `model`. Trace how the channel count and spatial dimensions change.
4. A fully connected layer from 256 input values to 4 outputs needs `256 * 4 + 4` parameters. Compare that with a 3 x 3 convolution from 1 input channel to 4 output channels, including biases.

## Important limitation

This notebook uses a hand-drawn synthetic image and an untrained model to demonstrate tensor operations. It is not an automotive perception system and makes no reliable vehicle, pedestrian, or lane predictions.